# Module 27 — Working Within Colab Pro

Module 31's real pretraining run happens on Google Colab Pro — this
project's chosen compute platform (see the root README's scope decision).
Colab Pro has real, fixed limits that a training run has to be designed
around:

- **$9.99/month**, ~100 compute units included (rolling, up to 90 days),
  pay-as-you-go top-ups at $0.10/unit beyond that.
- An **A100** burns ~15 units/hour — about **$1.50/hour** of equivalent
  compute, whether from the included allowance or a top-up.
- Sessions cap at **24 hours**, and a GPU isn't guaranteed to stay attached
  even within that window — Colab can disconnect on inactivity or reclaim
  the accelerator under demand.

The consequence: any run longer than one sitting **will** get interrupted
at some point. The only piece of this that's actually testable outside a
real Colab session — and the one that matters most — is making sure a
training run can be resumed from exactly where it left off, with zero lost
progress. That's what this module builds and verifies.

## 1. What a real checkpoint needs to contain

Saving just the model weights is not enough to resume *exactly* where
training left off. Also needed:
- **Optimizer state** — AdamW's per-parameter `m`/`v` running averages
  (Module 24). Restarting Adam from scratch mid-training re-introduces the
  exact instability warmup (Module 25) exists to avoid.
- **The RNG state** — if batches are drawn randomly, reproducing the
  training run requires reproducing the *same sequence* of random batches.
- **The step counter** — needed to resume the learning rate schedule
  (Module 25) at the right point, not restart it.

In [ ]:
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F


def save_checkpoint(model, optimizer, step):
    return {
        "model": copy.deepcopy(model.state_dict()),
        "optimizer": copy.deepcopy(optimizer.state_dict()),
        "rng_state": torch.get_rng_state().clone(),
        "step": step,
    }


def load_checkpoint(checkpoint, model, optimizer):
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    torch.set_rng_state(checkpoint["rng_state"])
    return checkpoint["step"]


print("Checkpoint save/load functions defined.")

## 2. The real test: does resuming produce IDENTICAL results?

Train for 5 steps, save a checkpoint, then compare two continuations for
the next 5 steps: one that just keeps running in the same process
(simulating an uninterrupted Colab session), and one that reloads
everything into a **completely fresh** model and optimizer from the saved
checkpoint (simulating "the session died and this is a new one, resuming
from Drive"). If checkpointing is done right, these must match exactly —
not approximately.

In [ ]:
def make_model_and_optimizer():
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 4))
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
    return model, optimizer


def train_step(model, optimizer):
    x = torch.randn(4, 8)
    y = torch.randint(0, 4, (4,))
    optimizer.zero_grad()
    loss = F.cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()
    return loss.item()


model, optimizer = make_model_and_optimizer()
for step in range(5):
    train_step(model, optimizer)

checkpoint = save_checkpoint(model, optimizer, step=5)

# "Session A": training just keeps going, uninterrupted
losses_uninterrupted = [train_step(model, optimizer) for _ in range(5)]
params_uninterrupted = [p.clone() for p in model.parameters()]

# "Session B": a brand new process, resuming from the saved checkpoint
model_resumed, optimizer_resumed = make_model_and_optimizer()  # fresh, random init - simulates a new session
resumed_step = load_checkpoint(checkpoint, model_resumed, optimizer_resumed)
losses_resumed = [train_step(model_resumed, optimizer_resumed) for _ in range(5)]
params_resumed = [p.clone() for p in model_resumed.parameters()]

assert resumed_step == 5
assert losses_uninterrupted == losses_resumed
for p1, p2 in zip(params_uninterrupted, params_resumed):
    assert torch.equal(p1, p2)

print("Losses after resuming:", losses_resumed)
print("Confirmed: resuming from a checkpoint into a brand-new model/optimizer produces BIT-IDENTICAL results to never having stopped at all.")

## Recap

- A real checkpoint needs model weights, optimizer state, RNG state, and
  the step counter — not just the weights.
- Verified directly: resuming from a checkpoint into a completely fresh
  model and optimizer (simulating a new Colab session after a disconnect)
  produces bit-identical losses and parameters to an uninterrupted run.
- Practically, for Module 31's real run: checkpoints must save to **Google
  Drive**, not Colab's local disk (which is wiped between sessions), and
  should save often enough that losing everything since the last
  checkpoint is an acceptable amount of A100-hours to redo — a real cost
  given the ~$1.50/hour figure above.

Phase 4 is complete: tokenization, data pipelines, mixed precision,
gradient accumulation, AdamW, learning rate schedules, gradient clipping,
and now checkpointing — everything needed to actually run training at
scale. Phase 5 covers generation itself (sampling, KV-caching) before the
real pretraining run.